# Первые эксперименты с математическими подсказками
Ноутбук воспроизводит анализ сохранённых ответов Qwen3-8B по методике v2. По умолчанию обращения к модели отключены. Полный набор: 9 случаев × 3 варианта × 2 seed = 54 результата и 72 вызова. Пилот v1 исключён из итоговых метрик.


In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'experiments/lab1/runner.py').exists()
sys.path.insert(0, str(ROOT / 'experiments/lab1'))
from runner import read, validate_cases, run
from analyze import main, collect
validate_cases(read('cases.json'))

## Данные и критерии
conceptual — нарушение или пропуск математического правила; arithmetic — вычислительная/алгебраическая ошибка после правильного выбора метода; none — корректное решение. Оцениваются номер первого ошибочного шага, тип ошибки, математическая корректность сообщения, отсутствие готового ответа и затраты. Только student_message показывается студенту. Поле internal может содержать решение. Ручные оценки находятся в evaluation.json; они не генерируются автоматически из правильности диагностики.


In [2]:
main()  # Только чтение журналов и пересчёт производных таблиц, без API
rows = collect()
print('Результатов:', len(rows))
print((ROOT / 'experiments/lab1/results/aggregate.json').read_text(encoding='utf-8'))

[
  {
    "variant": "A",
    "results": 18,
    "first_error_correct": 8,
    "error_type_correct": 6,
    "hint_correct": 17,
    "no_leak": 18,
    "calls": 18,
    "prompt_tokens": 6330,
    "output_tokens": 2452,
    "wall_seconds": 35.6707478999997,
    "false_alarms": 6
  },
  {
    "variant": "B",
    "results": 18,
    "first_error_correct": 10,
    "error_type_correct": 6,
    "hint_correct": 17,
    "no_leak": 18,
    "calls": 18,
    "prompt_tokens": 7126,
    "output_tokens": 2290,
    "wall_seconds": 33.232258899999124,
    "false_alarms": 6
  },
  {
    "variant": "C",
    "results": 18,
    "first_error_correct": 12,
    "error_type_correct": 8,
    "hint_correct": 18,
    "no_leak": 18,
    "calls": 36,
    "prompt_tokens": 14873,
    "output_tokens": 3962,
    "wall_seconds": 58.1688135000004,
    "false_alarms": 4
  }
]
Результатов: 54
[
  {
    "variant": "A",
    "results": 18,
    "first_error_correct": 8,
    "error_type_correct": 6,
    "hint_correct": 17,
    "

In [3]:
evaluated = json.loads((ROOT / 'experiments/lab1/results/evaluated.json').read_text(encoding='utf-8'))
for row in evaluated:
    if not all(row[k] for k in ['first_error_correct','error_type_correct','hint_correct','no_leak']):
        print(row['case_id'], row['variant'], row['seed'], row['student_message'], row['review_note'])

D2 A 42 Проверьте применение правила цепочки: сначала найдите производную внешней функции, а затем умножьте на производную внутренней функции. Правило цепочки сформулировано верно, но уже применено студентом; подсказка уводит от арифметической ошибки.
D2 B 42 Проверьте, как правильно раскрывать скобки при умножении на многочлен. Верно направляет к раскрытию скобок, несмотря на неверный тип во внутренней диагностике.
D2 C 42 Проверьте, правильно ли вы раскрываете скобки после применения правила дифференцирования. Верно направляет к раскрытию скобок, несмотря на неверный тип во внутренней диагностике.
D2 A 43 Проверьте, правильно ли применено правило цепочки при нахождении производной. Правило цепочки сформулировано верно, но уже применено студентом; подсказка уводит от арифметической ошибки.
D2 B 43 Проверьте, как правильно раскрывать скобки при умножении на многочлен. Верно направляет к раскрытию скобок, несмотря на неверный тип во внутренней диагностике.
D2 C 43 Проверьте, правильно л

## Повторный эксперимент
Новый запуск сохраняется отдельно и не перезаписывает журналы. Анализ использует только selection.json; для нового исследования необходимо изменить список выбранных запусков и заново оценить ответы вручную. Нужны работающая Ollama и модель с тем же digest. Фиксация seed не гарантирует побитовую воспроизводимость.


In [4]:
RUN_MODELS = False
if RUN_MODELS:
    run([c['id'] for c in read('cases.json')], repeat=2)

## Ограничения
Два seed и малый искусственный набор не позволяют делать статистически значимые выводы о превосходстве архитектуры. Порядок A/B/C фиксирован; загрузка модели, кэш и прогрев влияют на время. Подтверждение корректного решения следует отличать от общей просьбы перепроверить его. Педагогический эффект на реальных студентах не измерялся.
